# GAMER-TWIN: A Transformer-Based Digital Twin for Next-Action Prediction in Interactive Games
### Symbiosis Institute of Technology, Pune — Department of AIML
**Course:** Deep Learning Course Project  
**Students:** Mahit Turakhia, Het Shah  
**Faculty Guide:** Dr. Ananya Bhattacharjee  

---
## Notebook Overview
This notebook demonstrates the end-to-end GAMER-TWIN deep learning pipeline:
1. **Environment Setup & Seeding**
2. **Dataset Inspection & Validation Audit** (EgoCS-400K schema)
3. **Zero-Leakage Preprocessing & Group Splitting**
4. **Sequence Construction ($T=16 \to T+1$)**
5. **Deep Learning Model Architectures (4 Selectable Variants)**
6. **Training Pipeline & SSL Pretraining**
7. **Evaluation Metrics (Accuracy, Macro/Weighted F1, Top-3 Accuracy, Confusion Matrix)**
8. **Inference & Ranked Action Probabilities**

In [ ]:
# Step 1: Imports and Seeding
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in path
sys.path.append(os.path.abspath(".."))

from src.utils import load_config, set_seed, PROJECT_CLASSES, get_device
from src.data_loader import load_raw_egocs_data, generate_synthetic_gameplay_data
from src.data_validation import validate_dataset
from src.preprocessing import GameplayFeaturePreprocessor
from src.sequence_builder import prepare_train_val_test_splits
from src.models import create_model
from src.training import train_model
from src.evaluation import evaluate_model
from src.prediction import NextActionPredictor
from src.visualization import plot_class_distribution, plot_prediction_probabilities, plot_confusion_matrix

set_seed(42)
config = load_config("../config/config.yaml")
print(f"Device: {get_device()} | PyTorch: {torch.__version__} | Classes ({len(PROJECT_CLASSES)}): {PROJECT_CLASSES}")

In [ ]:
# Step 2: Load Telemetry and Run Validation Audit
data_dir = "../data/raw"
try:
    df, meta = load_raw_egocs_data(data_dir)
    print("Loaded Real EgoCS-400K dataset.")
except FileNotFoundError:
    print("Real dataset not found in data/raw/. Using verified synthetic test fixture for pipeline demonstration.")
    df = generate_synthetic_gameplay_data(num_rounds=5, ticks_per_round=400, output_dir="../data/sample_data", seed=42)
    meta = {"data_source": "SYNTHETIC_TEST_FIXTURE", "is_synthetic": True, "raw_record_count": len(df), "columns": list(df.columns)}

is_valid, report = validate_dataset(df, meta, config)
print(f"Validation Status: {report['status']} | Usable Records: {report['total_records']}")
print("Target Class Distribution:", report["target_class_counts"])

In [ ]:
# Step 3: Zero-Leakage Preprocessing & Group Splitting
train_ds, val_ds, test_ds, pipe_report = prepare_train_val_test_splits(df, config, output_dir="../outputs/processed")
input_dim = train_ds.sequences.shape[2]
print(f"Sequence Tensor Shape: {train_ds.sequences.shape} (Batches, Window Size, Feature Dim)")
print(f"Split counts: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")

In [ ]:
# Step 4: Model Instantiation and Training (Transformer Primary Model)
model_name = "transformer"
trained_model, history = train_model(
    model_name=model_name,
    train_dataset=train_ds,
    val_dataset=val_ds,
    input_dim=input_dim,
    config=config,
    checkpoint_dir="../outputs/checkpoints",
    epochs=3
)
print(f"Best Val Loss: {history['best_val_loss']} at epoch {history['best_epoch']}")

In [ ]:
# Step 5: Test Evaluation & Metrics
eval_results = evaluate_model(
    model=trained_model,
    test_dataset=test_ds,
    config=config,
    model_name="Transformer_Primary",
    data_provenance=meta.get("data_source", "DEMO"),
    top_k=3
)
print("Evaluation Metrics:")
for k, v in eval_results["metrics"].items():
    print(f"  {k}: {v}")

In [ ]:
# Step 6: Prediction & Probability Ranking on Sample Sequence
predictor = NextActionPredictor(model=trained_model)
sample_seq, sample_target_idx = test_ds[0]
ground_truth_class = PROJECT_CLASSES[sample_target_idx.item()]

pred_res = predictor.predict_sequence(sample_seq.numpy(), ground_truth_action=ground_truth_class)
print(f"Predicted Action: {pred_res['predicted_action']} ({pred_res['confidence_percentage']}%) | Ground Truth: {ground_truth_class}")
print("Ranked Probabilities:")
for r in pred_res["ranked_probabilities"][:5]:
    print(f"  Rank {r['rank']}: {r['class_name']} ({r['percentage']} %)")